In [16]:
"""
Test script for LMaaS
"""
from openai import AzureOpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam

import config
from idam_token_generator import IDAMTokenGenerator


idam = IDAMTokenGenerator(
    config.IDAM_TOKEN_ENDPOINT,
    config.IDAM_APP_CLIENT_ID,
    config.IDAM_APP_CLIENT_SECRET,
    config.IDAM_LMAAS_APP_AUDIENCE
)


llm = AzureOpenAI(
        azure_endpoint = config.OPENAI_ENDPOINT,
        azure_deployment = config.OPENAI_DEPLOYMENT_MODEL,
        api_version = config.OPENAI_AZURE_API_VERSION,
        azure_ad_token = idam.get_idam_token()
    )


Expiry_time: %s 1761696391
exp_time : %s 2025-10-29 00:06:31+00:00
current_time : %s 2025-10-29 03:01:52.619848+00:00
JWT token is NOT VALID
Generating new token.


/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'idam.gehealthcloud.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


IDAM Access Token is generated


/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'idam.gehealthcloud.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


IDAM Exchange Access Token is generated


In [17]:
messages = [
    ChatCompletionSystemMessageParam(role="system", content="You are a helpful assistant."),
    ChatCompletionUserMessageParam(role="user", content="What is the capital of France?"),
]

response = llm.chat.completions.create(
        model = config.OPENAI_DEPLOYMENT_MODEL,
        messages = messages,
    )

print(response.choices[0].message.content)

Paris.


In [46]:
# load the annotations data
import json
annotation_path01 = "/qumulo/shared_data/aofei_summer/RegTok/data/BiomedParse_SegVQA_Diagnosis_30k.json"

annotations = []
# load jsonlines
with open(annotation_path01, 'r', encoding='utf-8') as f:
    annotations = json.load(f)

In [47]:
len(annotations), list(annotations.keys())[0]

(3035,
 '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/test_27')

In [48]:
image_H, image_W = 1024, 1024
def preprocess_annotations(annotation):
    processed = []
    processed_with_info = []
    for region in annotation['mask_annotations']:
        item = {
            "id": region['id'],
        }
        item_with_info = {
            "id": region['id'],
            "mask_file": region.get("mask_file", ""),
            "image_id": region.get("image_id", "")
        }

        processed_bbox = [
            round(region['bbox'][1] / image_W, 3),
            round(region['bbox'][0] / image_H, 3),
            round(region['bbox'][3] / image_W, 3),
            round(region['bbox'][2] / image_H, 3)
        ]
        # item['bbox'] = processed_bbox
        processed_sentences = []
        for sentence in region['sentences']:
            processed_sentences.append(sentence['raw'])
        item['sentences'] = processed_sentences
        processed.append(item)
        processed_with_info.append(item_with_info)
    return processed, processed_with_info

In [49]:
sampled_image_id = 0
processed_items = []
processed_items_with_info = []
for k in annotations:
    processed, processed_with_info = preprocess_annotations(annotation=annotations[k])
    annotation = annotations[k]
    image_masks = {
        "image_id": sampled_image_id,
        "masks": processed
    }
    processed_items.append(image_masks) 
    image_masks_info = {
        "image_id": sampled_image_id,
        "image_file": annotation.get("image_file", ""),
        "modality": annotation.get("modality", ""),
        "num_masks": len(processed),
        "mask_id": [(item['id'], item['mask_file']) for item in processed_with_info]
    }

    processed_items_with_info.append(image_masks_info)
    sampled_image_id += 1

In [50]:
processed_items[1], processed_items_with_info[1]

({'image_id': 1, 'masks': [{'id': 148, 'sentences': ['benign tumor']}]},
 {'image_id': 1,
  'image_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/test/benign (89)_ultrasound_breast.png',
  'modality': 'US',
  'num_masks': 1,
  'mask_id': [(148,
    '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/test_mask/benign (89)_ultrasound_breast_benign+tumor.png')]})

In [ ]:
"""
**Notice:**
- There will be images with multiple abnormality masks, in this case, the diagnositic question should not be a general question about diagnosis of the entire image but should be specific to each abnormality mask. This means you should give enough context in each question to clearly indicate which abnormality mask it refers to.
- If you think it's impossible to generate separate questions,
"""

In [63]:
Alignment_prompt = """
You are an expert radiologist and dataset curator. 
You are given region-level annotations for one or more medical images. Each image item includes:
- "image_id": an integer id,
- "masks": a list of region objects, each with:
    - "id": id of this mask
    - "sentences": free-text descriptions (list of strings).

Assume you can see the image implicitly and must use only the provided annotation information (sentences) to perform the tasks below. 

Task (produce a single JSON output per image):
- Generate evaluation data in the form of VQA about the image and segmentation based on the provided region information.
Follow the style of radiology and clinical reasoning, and make sure the generated questions are natural, factual, and unambiguous.

Output rules and format (strict):
- Always return a JSON list containing one object per input image: [{...}, {...}, ...].
- Each image object must contain:
  {
    "image_id": <the input image_id>,
    "QAs": [ <list of Q&A objects> ]
  }
- Each dialogue turn is a JSON object:
  {
    "User": "<user question>",
    "Assistant": "<assistant answer>",
    "mask_ids": [ <list of mask indices appearing in the answer> ],
    "Question_type": "open" or "close"
  }

QA generation rules:
- For each image, generate one diagnostic Q&A pair for each provided mask.
- Each QA serves with 2 evaluation targets: VQA and segmentation.
- For each QA, include a short version of answer for the convenience of evaluation VQA (short diagnosis for open-ended VQA).
- For the mask with only organ and without abnormalities, you should skip it by generating empty QAs.
- For the mask with abnormalities, you may use open-ended questions to ask the diagnosis.
- Always include the corresponding mask_id(s) in the output key "mask_ids", do not include it in the ground truth answer.

**Notice:**
- There will be images with multiple abnormality masks, in this case, please identify the central or most important one (some of them might be one object) for the diagnosis question, if you can not identify such one, just leave the QAs as empty.
- In segmentation prompts, please explicitly use words like "segment xxx" or "segmentations" to let the model know segmentation is required.

**One special case for brain tumor MRI: **
- there may be three masks including enhancing tumor, non-enhancing tumor and tumor core. In this case, your answer should be brain tumor (incuding non-enhancing and enhancing tumor), and the recorded "mask_ids" should be a (sub) list of these three mask ids(e.g., [id1, id2, id3]) as one item in the list[[id1, id2, id3]].
- Focus on whole (overall) diagnosis if there are multiple abnormality masks, for example, if there are both enhancing, non-enhancing tumors and whole tumor, you should focus on whole tumor.

Example input of one image (for reference only):
{
  "image_id": 0,
  "masks": [
    {"id": 0, "sentences": ["liver"]},
    {"id": 1, "sentences": ["tumor"]},
    {"id": 2, "sentences": ["spleen"]},
  ]
}

Example output (for one image):
[
  {
    "image_id": 0,
    "QAs": [
      {
        "User": "What abnormality is seen on the liver? Please do diagnosis and then segment it if it exists.",
        "Assistant": "There is a tumor at the center right lobe, segmented as <mask> inside the liver.",
        "mask_ids": [1],
        "short answer": "Tumor",
        "Question_type": "open"
      },
    ]
  }
]

Please do not mention words like "annotations", "annotated regions" in the generated QA.

Finally: The API will provide the "masks" list as input. Produce the JSON outputs (one object per image) strictly following the rules above. Do not include any extra text outside the JSON list in the model's final reply. """


In [51]:
list_outputs = []
# list_outputs = list_outputs[:100]

In [52]:
output_json_file = "BiomedParse_SegVQA_GPT_Open_v2.jsonl"
ans_file = open(output_json_file, "a")

In [53]:
len(processed_items), processed_items[0]

(3035, {'image_id': 0, 'masks': [{'id': 54, 'sentences': ['benign tumor']}]})

In [59]:
processed_items[1267]

{'image_id': 1267,
 'masks': [{'id': 5618,
   'sentences': ['edema in brain magnetic resonance imaging',
    'edema in brain MRI',
    'edema in brain MR']},
  {'id': 5621, 'sentences': ['tumor core in brain MRI']},
  {'id': 5623, 'sentences': ['whole tumor']}]}

In [ ]:
batch_size = 10
max_retry = 3
from tqdm import tqdm
for i in tqdm(range(0, len(processed_items), batch_size)):
# for i in tqdm(range(1267, 1267+10, batch_size)):
    num_try = 1
    items = processed_items[i:i + batch_size]
    before_process_items = processed_items_with_info[i:i + batch_size]
    messages = [
        ChatCompletionSystemMessageParam(role="system", content="You are a helpful assistant." + Alignment_prompt + "\n\n"),
        ChatCompletionUserMessageParam(role="user", content="The input with multiple images:" + str(items)),
    ]
    try:
        response = llm.chat.completions.create(
                model = config.OPENAI_DEPLOYMENT_MODEL,
                messages = messages,
            )
    except:
        idam = IDAMTokenGenerator(
            config.IDAM_TOKEN_ENDPOINT,
            config.IDAM_APP_CLIENT_ID,
            config.IDAM_APP_CLIENT_SECRET,
            config.IDAM_LMAAS_APP_AUDIENCE
        )


        llm = AzureOpenAI(
                azure_endpoint = config.OPENAI_ENDPOINT,
                azure_deployment = config.OPENAI_DEPLOYMENT_MODEL,
                api_version = config.OPENAI_AZURE_API_VERSION,
                azure_ad_token = idam.get_idam_token()
            )
        response = llm.chat.completions.create(
                model = config.OPENAI_DEPLOYMENT_MODEL,
                messages = messages,
            )
        num_try += 1
        if num_try > max_retry:
            
            print(f"Max retries exceeded for batch starting at index {i}")
            continue

    # print(response.choices[0].message.content)
    llm_out = response.choices[0].message.content
    llm_out_json = json.loads(llm_out)
    items = processed_items[i:i + batch_size]
    original_items = processed_items_with_info[i:i + batch_size]
    for j in range(batch_size):
        original_item = original_items[j]
        llm_out_json[j]['image_file'] = original_item['image_file']
        llm_out_json[j]['mask_id'] = original_item['mask_id']
        llm_out_json[j]['modality'] = original_item['modality']
    list_outputs.extend(llm_out_json)
    ans_file.write("\n".join([json.dumps(x) for x in llm_out_json]) + "\n")
    ans_file.flush()
# ans_file.close()


  0%|          | 0/1 [00:00<?, ?it/s]

Expiry_time: %s 1761717347
exp_time : %s 2025-10-29 05:55:47+00:00
current_time : %s 2025-10-29 05:55:48.348944+00:00
JWT token is NOT VALID
Generating new token.


/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'idam.gehealthcloud.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


IDAM Access Token is generated


/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'idam.gehealthcloud.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


IDAM Exchange Access Token is generated


100%|██████████| 1/1 [00:56<00:00, 56.25s/it]


In [66]:
llm_out_json

[{'image_id': 1267,
  'QAs': [{'User': 'What is the overall diagnosis on this brain MRI? Please diagnose and then segment the whole tumor if present.',
    'Assistant': 'Findings are consistent with a brain tumor; the whole tumor is segmented as <mask>.',
    'mask_ids': [5623],
    'short answer': 'Brain tumor',
    'Question_type': 'open'}],
  'image_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task01_BrainTumour/test/BRATS_448_120_MRI-FLAIR_brain.png',
  'mask_id': [(5618,
    '/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task01_BrainTumour/test_mask/BRATS_448_120_MRI-FLAIR_brain_edema.png'),
   (5621,
    '/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task01_BrainTumour/test_mask/BRATS_448_120_MRI-FLAIR_brain_tumor+core.png'),
   (5623,
    '/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task01_BrainTumour/test_mask/BRATS_448_120_MRI-FLAIR_brain_whole+tumor.png')],
  'modality': 'MRI'},
 {'image_id': 1268,
  'QAs': [{'User': 

In [31]:
llm_out_json

[{'image_id': 0,
  'QAs': [{'User': 'On this breast ultrasound, what is the diagnosis of the visible lesion? After diagnosis, please segment the lesion.',
    'Assistant': 'Findings are consistent with a benign breast tumor; segmented as <mask> within the breast.',
    'mask_ids': [54],
    'short answer': 'Benign breast tumor',
    'Question_type': 'open'},
   {'User': 'What malignant abnormality is present in the breast on ultrasound, and then segment it?',
    'Assistant': 'A malignant breast tumor (carcinoma) is present; segmented as <mask> in the breast.',
    'mask_ids': [55],
    'short answer': 'Malignant breast tumor (carcinoma)',
    'Question_type': 'open'}],
  'image_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/test/benign (46)_ultrasound_breast.png',
  'mask_id': [(54,
    '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/test_mask/benign (46)_ultrasound_breast_benign+tumor.png'),
   (55,
    '/qumulo/shared_data/aofei_summ